#### SqliteSaver로 메모리 영구화하기, 파이썬 설치 시 자동으로 셋팅 되므로 별도의 추가 설정이 필요 없음


In [2]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain.agents import create_agent
import sqlite3


In [3]:
load_dotenv()

model = ChatOpenAI(model='gpt-5-nano')

# 데이터베이스 연결 및 SqliteSaver 초기화
conn = sqlite3.connect('memory.db', check_same_thread=False) # default값이 true 이므로 반드시 False 옵션 설정
checkpointer = SqliteSaver(conn)

agent = create_agent(
    model=model,
    checkpointer=checkpointer   # checkpointer : agent는 대화의 맥락을 저장하고 보관
)


In [3]:
# 사용자 식별을 위한 thread_id 설정
user_id = 'user_123'
config = {'configurable': {'thread_id': user_id}}  # configurable 환경변수 

# 1. 첫 번째 대화: 이름 정보 제공
query1 = '내 이름은 철수야.'
print(f'사용자: {query1}')

result = agent.invoke(
    {'messages': [{'role': 'user', 'content': query1}]}, 
    config
)

print(f'Agent: {result['messages'][-1].content}')  # agent는 응답을 생성



사용자: 내 이름은 철수야.
Agent: 반갑습니다, 철수님! 이 대화에서만 이름으로 부르겠습니다. 어떤 도움이 필요하신가요? 예를 들면 번역, 글쓰기, 정보 찾아드리기, 계획 짜기 등 모든 것이 가능해요. 무엇을 도와드릴까요?


In [8]:
user_id = 'user_123'
config = {'configurable': {'thread_id': user_id}}  # configurable 환경변수 
 

In [9]:
# 2. 두 번째 대화: 기억력 테스트 (같은 thread_id 사용)
# checkpointer 장착 -> 대화의 맥락을 기억

query2 = '내 이름이 뭐라고 했어?'
print(f'사용자: {query2}')

result = agent.invoke(
    {'messages': [{'role': 'user', 'content': query2}]}, 
      config          # config = {'configurable': {'thread_id': user_id}}
)

print(f'Agent: {result['messages'][-1].content}')  # agent는 응답을 생성


사용자: 내 이름이 뭐라고 했어?
Agent: 철수라고 하셨습니다. 다른 도움이 필요하신가요?


In [5]:
# 3. 세 번째 대화: 다른 사용자로 설정(다른 thread_id 사용)
new_user_id = 'user_789'
new_config = {'configurable': {'thread_id': new_user_id}}

query3 = '내 이름이 뭐라고 했어?'  # user_789의 첫 번째 대화이므로 대화의 내용을 기억 못함
print(f'사용자: {query3}')

result = agent.invoke(
    {'messages': [{'role': 'user', 'content': query3}]}, 
    new_config          # config = {'configurable': {'thread_id': user_id}}
)

print(f'Agent: {result['messages'][-1].content}')  # agent는 응답을 생성


사용자: 내 이름이 뭐라고 했어?
Agent: 죄송하지만 이 대화에선 당신의 이름을 들어본 적이 없어요. 이름이 무엇인지 다시 알려주실 수 있나요? 그 이름으로 불러드릴게요.
